In [1]:
%pip install xgboost scikit-learn

   ---------------------------------------- 0.0/48.9 MB ? eta -:--:--
   ---------------------------------------- 0.3/48.9 MB ? eta -:--:--
    --------------------------------------- 0.8/48.9 MB 2.3 MB/s eta 0:00:21
    --------------------------------------- 1.0/48.9 MB 2.2 MB/s eta 0:00:23
   - -------------------------------------- 1.6/48.9 MB 2.0 MB/s eta 0:00:25
   - -------------------------------------- 1.8/48.9 MB 1.8 MB/s eta 0:00:27
   - -------------------------------------- 2.4/48.9 MB 1.9 MB/s eta 0:00:25
   -- ------------------------------------- 2.9/48.9 MB 2.0 MB/s eta 0:00:23
   -- ------------------------------------- 3.4/48.9 MB 2.0 MB/s eta 0:00:23
   --- ------------------------------------ 4.2/48.9 MB 2.2 MB/s eta 0:00:20
   ---- ----------------------------------- 5.0/48.9 MB 2.4 MB/s eta 0:00:19
   ---- ----------------------------------- 6.0/48.9 MB 2.6 MB/s eta 0:00:17
   ----- ---------------------------------- 6.6/48.9 MB 2.6 MB/s eta 0:00:17
   ----- ----


[notice] A new release of pip is available: 26.1.2 -> 26.3
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
import pandas as pd
from xgboost import XGBClassifier
from sklearn.metrics import (
    average_precision_score, classification_report, confusion_matrix
)

train = pd.read_parquet("../dataset/train.parquet")
test = pd.read_parquet("../dataset/test.parquet")

FEATURES = [
    "step", "amount", "oldbalanceOrg", "oldbalanceDest",
    "is_transfer", "amount_to_balance", "drains_account",
    "orig_zero_balance", "dest_zero_balance",
    "orig_prev_count", "dest_prev_count",
    "amount_vs_orig_avg", "dest_prev_received",
]
TARGET = "isFraud"

X_train, y_train = train[FEATURES], train[TARGET]
X_test, y_test = test[FEATURES], test[TARGET]

# Give fraud rows more weight, by the imbalance ratio
spw = (y_train == 0).sum() / (y_train == 1).sum()
print("scale_pos_weight:", round(spw, 1))

model = XGBClassifier(
    n_estimators=300,
    max_depth=6,
    learning_rate=0.1,
    scale_pos_weight=spw,
    eval_metric="aucpr",
    tree_method="hist",
    n_jobs=-1,
    random_state=42,
)
model.fit(X_train, y_train)

proba = model.predict_proba(X_test)[:, 1]
print("PR-AUC:", round(average_precision_score(y_test, proba), 4))
print("Random baseline:", round(y_test.mean(), 4))

pred = (proba >= 0.5).astype(int)
print(confusion_matrix(y_test, pred))
print(classification_report(y_test, pred, digits=3))

scale_pos_weight: 411.3
PR-AUC: 0.9983
Random baseline: 0.0367
[[41926    24]
 [    6  1594]]
              precision    recall  f1-score   support

           0      1.000     0.999     1.000     41950
           1      0.985     0.996     0.991      1600

    accuracy                          0.999     43550
   macro avg      0.993     0.998     0.995     43550
weighted avg      0.999     0.999     0.999     43550



In [3]:
imp = pd.Series(model.feature_importances_, index=FEATURES).sort_values(ascending=False)
print(imp.round(4))

drains_account        0.5175
amount_to_balance     0.4381
orig_zero_balance     0.0280
amount                0.0093
dest_zero_balance     0.0019
oldbalanceDest        0.0012
step                  0.0009
is_transfer           0.0008
oldbalanceOrg         0.0008
dest_prev_received    0.0007
dest_prev_count       0.0007
amount_vs_orig_avg    0.0000
orig_prev_count       0.0000
dtype: float32


In [5]:
import json, os

os.makedirs("../models", exist_ok=True)
model.save_model("../models/xgb_baseline.json")

with open("../models/features.json", "w") as f:
    json.dump(FEATURES, f)

print(os.listdir("../models"))

['features.json', 'xgb_baseline.json']
